In [4]:
import pandas as pd
from sqlalchemy import text

from data_access.connection import engine
from data_access.config import project_path


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

## Создание DF на основе данных при инициализации задачи (то есть, момент, когда задача была создана и была заполнена информация о ее приоритете и т.д.)

Проблемой оказалось то, что приоритет на задачу могли установить не в момент ее инициализации, а на последующей итерации

In [5]:
query_path = project_path.query("training_dataset.sql")
query = text(query_path.read_text(encoding="utf-8"))

issue_df = pd.read_sql_query(query, engine)
issue_df

,ID,Project_ID,Creation_Date,Description_Issue_Length,Duration_Days,Initial_Priority,Initial_Type,Description_Project_Length
0,85,1,2016-02-24 10:51:02,249,303.1658,Minor,Bug,146
1,86,1,2016-02-23 18:31:38,125,303.8457,Minor,Improvement,146
2,89,1,2016-02-22 15:27:31,384,0.1347,Minor,Story,146
3,90,1,2016-02-17 14:28:31,18,3.4999,Minor,Bug,146
4,91,1,2016-02-16 14:59:06,579,3.2120,Minor,Improvement,146
...,...,...,...,...,...,...,...,...
251915,509031,33,2009-04-09 09:15:20,68,96.3001,Major,Improvement,130
251916,509032,33,2009-04-09 09:14:27,192,322.5195,Major,Improvement,130
251917,509033,33,2009-04-09 09:02:06,107,48.2127,Minor - P4,Bug,130
251918,509034,33,2009-04-09 08:59:26,0,138.1045,Major - P3,Improvement,130


## Проверка структуры итогового датасета

In [6]:
structure_summary = pd.Series({
    "row_count": len(issue_df),
    "unique_id_count": issue_df["ID"].nunique(),
    "duplicate_id_count": issue_df["ID"].duplicated().sum(),
})

structure_summary

row_count             251920
unique_id_count       251920
duplicate_id_count         0
dtype: int64

In [7]:
missing_values = issue_df.isna().sum().sort_values(ascending=False)

missing_values

ID                            0
Project_ID                    0
Creation_Date                 0
Description_Issue_Length      0
Duration_Days                 0
Initial_Priority              0
Initial_Type                  0
Description_Project_Length    0
dtype: int64

In [8]:
duration_quality = pd.Series({
    "negative_duration_count": issue_df["Duration_Days"].lt(0).sum(),
    "zero_duration_count": issue_df["Duration_Days"].eq(0).sum(),
    "missing_duration_count": issue_df["Duration_Days"].isna().sum(),
})

duration_quality

negative_duration_count    0
zero_duration_count        0
missing_duration_count     0
dtype: int64

In [9]:
issue_df["Duration_Days"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)

count    251920.000000
mean         50.691961
std          79.372417
min           0.003500
50%          13.909100
90%         170.121390
95%         248.978675
99%         338.139316
max         364.999800
Name: Duration_Days, dtype: float64

In [11]:
category_summary = issue_df[[
    "Project_ID",
    "Initial_Type",
    "Initial_Priority",
]].nunique()

category_summary

Project_ID          39
Initial_Type        69
Initial_Priority    19
dtype: int64

In [13]:
issue_df["Initial_Priority"].value_counts(dropna=False)

Initial_Priority
Minor             47167
Major - P3        46577
Unknown           33229
Medium            31046
Low               27236
Major             22131
None              12204
High               9193
To be reviewed     7294
Critical           4152
Minor - P4         3242
Blocker            2441
Critical - P2      1503
Highest            1502
Trivial            1403
Trivial - P5        608
Blocker - P1        573
Undefined           306
Lowest              113
Name: count, dtype: int64

In [14]:
issue_df["Initial_Type"].value_counts(dropna=False)

Initial_Type
Bug               132485
Improvement        32926
Task               25207
Story              24412
New Feature        11568
                   ...  
Action Item            1
IT Help                1
Module Request         1
Workload               1
Change                 1
Name: count, Length: 69, dtype: int64

## Хронологическое разделение данных

In [15]:
issue_df = issue_df.sort_values(
    ["Creation_Date", "ID"]
).reset_index(drop=True)

train_end = int(len(issue_df) * 0.70)
validation_end = int(len(issue_df) * 0.85)

train_df = issue_df.iloc[:train_end].copy()
validation_df = issue_df.iloc[train_end:validation_end].copy()
test_df = issue_df.iloc[validation_end:].copy()

In [16]:
split_summary = pd.DataFrame({
    "train": {
        "row_count": len(train_df),
        "creation_date_min": train_df["Creation_Date"].min(),
        "creation_date_max": train_df["Creation_Date"].max(),
    },
    "validation": {
        "row_count": len(validation_df),
        "creation_date_min": validation_df["Creation_Date"].min(),
        "creation_date_max": validation_df["Creation_Date"].max(),
    },
    "test": {
        "row_count": len(test_df),
        "creation_date_min": test_df["Creation_Date"].min(),
        "creation_date_max": test_df["Creation_Date"].max(),
    },
}).T

split_summary

,row_count,creation_date_min,creation_date_max
train,176344,2003-04-08 00:48:51,2017-04-24 18:28:43
validation,37788,2017-04-24 18:38:00,2018-11-08 14:59:22
test,37788,2018-11-08 15:01:59,2020-10-22 15:16:24


In [17]:
duration_distribution = pd.DataFrame({
    "train": train_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "validation": validation_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
    "test": test_df["Duration_Days"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ),
}).T

duration_distribution

,count,mean,std,min,50%,90%,95%,99%,max
train,176344.0,52.130042,82.123674,0.0035,13.05515,181.79215,257.990140,341.099980,364.9998
validation,37788.0,50.244721,78.511083,0.0035,14.53965,166.91898,246.043345,339.515858,364.9874
test,37788.0,44.428156,65.678708,0.0035,16.83905,128.46751,191.898300,313.355377,364.9152
